In [ ]:
# ==========================
# 1_data_preprocessing.ipynb
# ==========================

!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -q torchinfo split-folders einops

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import random
import splitfolders

torch.manual_seed(42)
torch.cuda.empty_cache()

# Load YOLO model
yolo = torch.hub.load('ultralytics/yolov5', 'yolov5s')
yolo.conf = 0.5

def infer_crop(image_path: str):
    with torch.inference_mode():
        image = Image.open(image_path).convert('RGB')
        results = yolo(image)
        boxes = results.xyxy[0].cpu()
        imgs = []
        if len(boxes) == 0:
            return []
        else:
            for box in boxes:
                x1, y1, x2, y2, *_ = box.tolist()
                imgs += [image.crop((int(x1), int(y1), int(x2), int(y2)))]
            return imgs

# Mask generation
def generate_mask(image:torch.Tensor):
    PATCH_SIZE = 16
    NUM_PATCH = int(224**2 / 16**2)
    NUM_PER_LENGTH = int(NUM_PATCH**0.5)
    PORTION = 0.75
    mask = torch.zeros(image.shape)
    pos = torch.randperm(NUM_PATCH)
    pos = pos[:int(NUM_PATCH*(1- PORTION))]
    for i in pos:
        top = (i // NUM_PER_LENGTH)*PATCH_SIZE
        left = (i % NUM_PER_LENGTH)*PATCH_SIZE
        mask[:, top:top+PATCH_SIZE, left:left+PATCH_SIZE] = 1
    return mask

# Dataset classes
class KneePerClassDataset(Dataset):
    def __init__(self, root_dir:str, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.label = int(self.root_dir.split('/')[-1])
        self.image_paths = self._get_image_paths()
        self.mask_fn = generate_mask
        self.normalize_fn = transforms.Normalize((0.5,), (0.5,))
    def _get_image_paths(self):
        paths = []
        for filename in os.listdir(self.root_dir):
            if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
                paths.append(os.path.join(self.root_dir, filename))
        return paths
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, index):
        image = Image.open(self.image_paths[index])
        label = self.label
        if self.transform: image = self.transform(image)
        masked_image = image*self.mask_fn(image)
        image = self.normalize_fn(image)
        masked_image = self.normalize_fn(masked_image)
        return [image, label, masked_image]

class KneeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes, self.class_to_idx = self._find_classes()
        self.image_paths = self._get_image_paths()
        self.mask_fn = generate_mask
        self.normalize_fn = transforms.Normalize((0.5,), (0.5,))
    def _find_classes(self):
        classes = sorted([d.name for d in os.scandir(self.root_dir) if d.is_dir()])
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx
    def _get_image_paths(self):
        paths = []
        for class_name in self.classes:
            class_dir = os.path.join(self.root_dir, class_name)
            for filename in os.listdir(class_dir):
                if filename.lower().endswith((".jpg", ".png", ".jpeg")):
                    paths.append((os.path.join(class_dir, filename), self.class_to_idx[class_name]))
        return paths
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, index):
        image, target = self.image_paths[index]
        image = Image.open(image)
        if self.transform: image = self.transform(image)
        masked_image = image*self.mask_fn(image)
        image = self.normalize_fn(image)
        masked_image = self.normalize_fn(masked_image)
        return [image, target, masked_image]

# Example transforms and dataloaders
BATCH_SIZE = 8
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomEqualize(1),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])
dataset_path = "/PATH/TO/DATA"

train_dataset = KneeDataset(f'{dataset_path}/train', transform=train_transform)
val_dataset = KneeDataset(f'{dataset_path}/val', transform=train_transform)

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True),
}

# Visualize a batch
invTrans = transforms.Compose([
    transforms.Normalize(mean = [0.], std = [1/0.5]),
    transforms.Normalize(mean = [-0.5], std = [1.])
])

images, _, masked_images = next(iter(dataloaders['train']))
images, masked_images = invTrans(images), invTrans(masked_images)
data_grid = torchvision.utils.make_grid(images)
masked_data_grid = torchvision.utils.make_grid(masked_images)
plt.figure(figsize=(20, 10))
plt.subplot(2, 1, 1)
plt.imshow(data_grid.permute(1,2,0)); plt.axis('off'); plt.title('Original')
plt.subplot(2, 1, 2)
plt.imshow(masked_data_grid.permute(1,2,0)); plt.axis('off'); plt.title('Masked')
plt.show()